# 07 — Low-Rank RNN for Cognitive Tasks (Paradigm A)

**References:**
- Dubreuil, A., Valente, A., Beiran, M., Mastrogiuseppe, F., & Ostojic, S. (2022).
  *The role of population structure in computations through neural dynamics.* Nature Neuroscience, 25, 783-794.
- Valente, A., Ostojic, S., & Bhatt, D. (2022).
  *Extracting computational mechanisms from neural data using low-rank RNNs.* NeurIPS.

**Plan:**
1. **Tutorial** (D2022_10): Rank-1 RDM training, concepts, connectivity, population analysis, resampling
2. **Mante Full Analysis** (fig4-5): Rank-1 Mante training, psychometrics, state-space plots, gain analysis, effective input, overlap matrices
3. **All 5 Tasks**: Training check on RDM, Romo, Raposo, DMS, Mante
4. **Rank Comparison**: Rank 1 vs 2 vs 3 on oscillator dataset

## 0. Setup

All imports are from the **neuralrnn** framework. Analysis/plotting helpers are defined
locally in this notebook to keep the framework focused on reusable components.

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../src')

import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import torch
import torch.nn as nn
from math import sqrt
from scipy import stats
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.decomposition import PCA

# ─── NeuralRNN framework (AutoModel / AutoConfig pattern) ───
from neuralrnn import AutoConfig, AutoModel, Trainer, TrainingArguments
from neuralrnn import SupervisedObjective, load_dataset
from neuralrnn.data import CognitiveTaskDataset

# ─── Reusable loss functions ───
from neuralrnn.losses import loss_mse, accuracy_general

# ─── Analysis utilities ───
from neuralrnn.analysis.linalg_utils import phi_prime, gram_factorization, overlap_matrix
from neuralrnn.analysis.population_structure import make_vecs, gmm_fit

# ─── Task constants (only mante — needed for analysis helper functions) ───
from neuralrnn.data.tasks.lr_mante_task import (
    total_duration as _mante_T,
    stim_begin as _mante_stim_begin,
    stim_end as _mante_stim_end,
    response_begin as _mante_response_begin,
    fixation_discrete as _mante_fixation,
    SCALE as _mante_SCALE,
    SCALE_CTX as _mante_SCALE_CTX,
    STD_DEFAULT as _mante_STD,
    DELTA_T as _mante_deltaT,
)

print('✓ All imports successful — using neuralrnn AutoModel + Trainer')

### 0.1 Local Helper Functions

Training, plotting, and analysis utilities ported from the reference codebase.
These are notebook-local to keep the framework generic.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Notebook-local plotting helpers
# ═══════════════════════════════════════════════════════════════════

def setup_matplotlib():
    """Set matplotlib defaults matching the reference publication style."""
    plt.rcParams['axes.titlesize'] = 24
    plt.rcParams['axes.labelsize'] = 19
    plt.rcParams['xtick.labelsize'] = 16
    plt.rcParams['ytick.labelsize'] = 16
    plt.rcParams['figure.figsize'] = (6, 4)
    plt.rcParams['axes.titlepad'] = 24
    plt.rcParams['axes.labelpad'] = 10
    plt.rcParams['axes.spines.top'] = False
    plt.rcParams['axes.spines.right'] = False
    plt.rcParams['font.size'] = 14
    plt.rcParams['legend.fontsize'] = 'medium'


def center_axes(ax):
    """Center axes at origin with no ticks."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_position('zero')
    ax.spines['left'].set_position('zero')
    ax.set(xticks=[], yticks=[])


def get_lower_tri_heatmap(ov, cbar=False, cbar_shrink=0.9, cbar_pad=0.3,
                          figsize=None, ax=None):
    """Plot lower-triangular heatmap of an overlap matrix."""
    import matplotlib.ticker as ticker
    mask = np.zeros_like(ov, dtype=bool)
    mask[np.triu_indices_from(mask)] = True
    ov[np.diag_indices_from(ov)] = 0
    mask = mask.T
    if figsize is None:
        figsize = matplotlib.rcParams['figure.figsize']
    bound = max(abs(np.min(ov)), abs(np.max(ov)))
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    cmap = sns.diverging_palette(220, 10, sep=10, as_cmap=True)
    if not cbar:
        sns.heatmap(ov[:-1, 1:], mask=mask[:-1, 1:], cmap=cmap, center=0,
                    square=True, linewidths=0.5, cbar=False,
                    vmin=-bound, vmax=bound, ax=ax)
    else:
        sns.heatmap(ov[:-1, 1:], mask=mask[:-1, 1:], cmap=cmap, center=0,
                    square=True, linewidths=0.5, cbar=True,
                    vmin=-bound, vmax=bound, ax=ax,
                    cbar_kws={"shrink": cbar_shrink,
                              "ticks": ticker.MaxNLocator(3),
                              'pad': cbar_pad})
    ax.xaxis.tick_top()
    ax.yaxis.tick_right()
    return ax


def pop_scatter_linreg(vec1, vec2, pops, n_pops=None, linreg=True,
                       colors=('blue', 'green', 'red', 'violet', 'gray'),
                       figsize=(5, 5), size=10., ax=None, ls=None, lw=None):
    """Scatter plot of (vec1, vec2) separated by population with optional
    linear regression lines."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    center_axes(ax)
    xmax = max(abs(vec1.min()), vec1.max())
    xmin = -xmax
    ax.set_xlim(xmin - .1*(xmax-xmin), xmax + .1*(xmax-xmin))
    ymax = max(abs(vec2.min()), vec2.min())
    ymin = -ymax
    ax.set_ylim(ymin - .1*(ymax-ymin), ymax + .1*(ymax-ymin))
    xs = np.linspace(xmin - .1*(xmax-xmin), xmax + .1*(xmax-xmin), 100)
    if n_pops is None:
        n_pops = np.unique(pops).shape[0]
    for i in range(n_pops):
        ax.scatter(vec1[pops == i], vec2[pops == i],
                   color=colors[i], s=size)
        if linreg:
            slope, intercept, r, p, std_err = stats.linregress(
                vec1[pops==i], vec2[pops==i])
            print(f"pop {i}: slope={slope:.2f}, intercept={intercept:.2f}")
            ax.plot(xs, slope * xs + intercept, color=colors[i],
                    zorder=-1, ls=ls, lw=lw)
    ax.set_xticks([])
    ax.set_yticks([])


# ═══════════════════════════════════════════════════════════════════
# Task-specific helpers (notebook-local — analysis only)
# ═══════════════════════════════════════════════════════════════════

def psychometric_matrices_mante(net, ax=None, coherences=None):
    """Plot 2x1 psychometric matrices for Mante task (color/motion contexts).

    Uses neuralrnn Mante task constants.
    """
    n_trials = 10
    if coherences is None:
        coherences = np.arange(-5, 6, 2)
    if ax is None:
        fig, ax = plt.subplots(2, 1, figsize=(5, 8))
    for ctx_idx, ctx in enumerate([1, 2]):
        mat = np.zeros((len(coherences), len(coherences)))
        for i, coh1 in enumerate(coherences):
            for j, coh2 in enumerate(coherences):
                inputs_sensory = _mante_STD * torch.randn(
                    (n_trials, _mante_T, 2), dtype=torch.float32)
                inputs_context = torch.zeros((n_trials, _mante_T, 2))
                inputs = torch.cat([inputs_sensory, inputs_context], dim=2)
                inputs[:, _mante_stim_begin:_mante_stim_end, 0] += coh1 * _mante_SCALE
                inputs[:, _mante_stim_begin:_mante_stim_end, 1] += coh2 * _mante_SCALE
                ctx_channel = 2 if ctx == 1 else 3
                inputs[:, _mante_fixation:_mante_response_begin, ctx_channel] = (
                    1.0 * _mante_SCALE_CTX)
                with torch.no_grad():
                    output = net.forward(inputs)
                    decisions = torch.sign(
                        output[:, _mante_response_begin:, :].mean(dim=1).squeeze())
                mat[len(coherences) - j - 1, i] = decisions.mean().item()
        im = ax[ctx_idx].matshow(mat, cmap='gray', vmin=-1, vmax=1)
        ax[ctx_idx].set_xticks([])
        ax[ctx_idx].set_yticks([])
    return ax


def to_support_net(net, z, new_size=None, take_means=False):
    """Convert a LowRankRNN to a SupportLowRankRNN based on clustering labels.

    Note: This imports SupportLowRankRNN from the reference project
    (not yet ported to neuralrnn).
    """
    from low_rank_rnns.modules import SupportLowRankRNN
    X = np.vstack(make_vecs(net)).transpose()
    _, counts = np.unique(z, return_counts=True)
    n_components = counts.shape[0]
    weights = counts / net.hidden_size
    if take_means:
        means = np.vstack([X[z == i].mean(axis=0) for i in range(n_components)])
    else:
        means = np.zeros((n_components, X.shape[1]))
    covariances = [np.cov(X[z == i].transpose()) for i in range(n_components)]
    rank = net.rank
    basis_dim = 2 * rank + net.input_size + net.output_size
    m_init = torch.zeros(rank, n_components, basis_dim)
    n_init = torch.zeros(rank, n_components, basis_dim)
    wi_init = torch.zeros(net.input_size, n_components, basis_dim)
    wo_init = torch.zeros(net.output_size, n_components, basis_dim)
    if new_size is None:
        new_size = net.hidden_size
    m_means = torch.from_numpy(means[:, :rank]).t() / sqrt(new_size)
    n_means = torch.from_numpy(means[:, rank:2*rank]).t() / sqrt(new_size)
    wi_means = torch.from_numpy(
        means[:, 2*rank:2*rank+net.input_size]
    ).t()
    for i in range(n_components):
        G = covariances[i]
        X_reduced = gram_factorization(G)
        for k in range(rank):
            m_init[k, i] = torch.from_numpy(X_reduced[k]) / sqrt(new_size)
            n_init[k, i] = torch.from_numpy(
                X_reduced[rank + k]
            ) / sqrt(new_size)
        for k in range(net.input_size):
            wi_init[k, i] = torch.from_numpy(X_reduced[2*rank + k])
        for k in range(net.output_size):
            wo_init[k, i] = torch.from_numpy(
                X_reduced[2*rank + net.input_size + k]
            ) / new_size
    net2 = SupportLowRankRNN(
        net.input_size, new_size, net.output_size,
        net.noise_std, net.alpha, rank,
        n_components, weights, basis_dim,
        m_init, n_init, wi_init, wo_init,
        m_means, n_means, wi_means)
    return net2


print('✓ All local helpers defined')

### 0.2 Task Data Helpers (analysis only — training uses `load_dataset`)

These thin wrappers around neuralrnn task generators provide the 6-return-value
(train/val split) API and per-condition trial generation used by analysis cells.

In [ ]:
# ─── Thin wrappers for single-trial / per-condition generation (analysis only) ───
from neuralrnn.data.tasks import rdm_trials, lr_mante_trials

def generate_rdm_data(num_trials, coherences=None, std=None,
                      fraction_validation_trials=0.2, fraction_catch_trials=0.):
    """Generate RDM data with train/val split (for analysis cells).
    Training should use load_dataset('rdm', ...).
    """
    if std is None: std = 0.1
    kwargs = dict(num_trials=num_trials, std=std, fraction_catch_trials=fraction_catch_trials)
    if coherences is not None: kwargs['coherences'] = coherences
    inputs, targets, mask, conditions = rdm_trials(**kwargs)
    if fraction_validation_trials > 0:
        split_at = num_trials - int(num_trials * fraction_validation_trials)
        return (inputs[:split_at], targets[:split_at], mask[:split_at],
                inputs[split_at:], targets[split_at:], mask[split_at:])
    return inputs, targets, mask

def generate_mante_data(num_trials, coherences=None, std=None,
                        fraction_validation_trials=0.2, fraction_catch_trials=0.,
                        coh_color_spec=None, coh_motion_spec=None, context_spec=None):
    """Generate Mante data with train/val split (for analysis cells)."""
    if std is None: std = 0.1
    kwargs = dict(num_trials=num_trials, std=std, fraction_catch_trials=fraction_catch_trials)
    if coherences is not None: kwargs['coherences'] = coherences
    if coh_color_spec is not None: kwargs['coh_color'] = coh_color_spec
    if coh_motion_spec is not None: kwargs['coh_motion'] = coh_motion_spec
    if context_spec is not None: kwargs['context'] = context_spec
    inputs, targets, mask, conditions = lr_mante_trials(**kwargs)
    if fraction_validation_trials > 0:
        split_at = num_trials - int(num_trials * fraction_validation_trials)
        return (inputs[:split_at], targets[:split_at], mask[:split_at],
                inputs[split_at:], targets[split_at:], mask[split_at:])
    return inputs, targets, mask

print('✓ Analysis task wrappers defined')

In [ ]:
# Style
setup_matplotlib()
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['figure.figsize'] = (7, 5)

---
## 1. Tutorial: Low-Rank RNN Fundamentals

### 1.1 Theory

A standard RNN with $N$ neurons has recurrence:
$$h_{t+1} = h_t + \alpha\left(-h_t + W^{rec} r_t + W^{in} x_t\right) + \sigma\xi_t$$

where $W^{rec} \in \mathbb{R}^{N \times N}$ has $N^2$ free parameters.

A **low-rank RNN** constrains the recurrent matrix:
$$W^{rec} = \frac{1}{N} m \, n^T = \frac{1}{N} \sum_{k=1}^R m^{(k)} \otimes n^{(k)}$$

with $m, n \in \mathbb{R}^{N \times R}$, $R \ll N$. This gives:

1. **Parameter efficiency**: $2NR$ instead of $N^2$
2. **Transparent dynamics**: Recurrent input $r \cdot n \cdot m^T$ lies in $\mathrm{span}(m)$
3. **Natural analysis**: The $m$ columns form an $R$-dimensional coordinate system
4. **Population structure**: Neurons cluster by their $(m_i, n_i, w^{in}_i, w^{out}_i)$ vectors

### 1.2 Model Instantiation

In [ ]:
# Create a rank-1 low-rank RNN using AutoConfig (framework pattern)
hidden_size = 500
cfg = AutoConfig.for_model('lowrank_rnn',
    input_dim=1, latent_dim=hidden_size, output_dim=1,
    rank=1, alpha=0.2, noise_std=0.05, activation='tanh',
)
model = AutoModel.from_config(cfg)
print(f'{model}')
print(f'  hidden_size={model.hidden_size}, rank={model.rank}')
print(f'  parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'  m shape={list(model.m.shape)}, n shape={list(model.n.shape)}')

### 1.3 RDM Task and Training

The Random Dot Motion (RDM) task: a noisy coherence signal is presented,
and the network must report whether the coherence is positive or negative.

In [ ]:
# Generate RDM data using load_dataset (framework pattern)
# Split into train/val manually for evaluation
ds_train = load_dataset('rdm', batch_size=32, num_trials=800, seed=42)
ds_val = load_dataset('rdm', batch_size=32, num_trials=200, seed=99)

# Grab validation tensors for manual eval
x_val = ds_val.inputs
y_val = ds_val.targets
mask_val = ds_val.mask

print(f'Train: {ds_train.inputs.shape}, Val: {x_val.shape}')

# Visualize one trial
fig, axes = plt.subplots(3, 1, figsize=(6, 4), sharex=True)
axes[0].plot(ds_train.inputs[0, :, 0], 'b', lw=1); axes[0].set_ylabel('Input')
axes[1].plot(ds_train.targets[0, :, 0], 'r', lw=1); axes[1].set_ylabel('Target')
axes[2].plot(ds_train.mask[0, :, 0], 'gray', lw=1); axes[2].set_ylabel('Mask')
plt.tight_layout(); plt.show()

In [ ]:
# Train using framework Trainer + SupervisedObjective (regression)
# ~500 steps ≈ 20 epochs with batch_size=32 and 800 trials
objective = SupervisedObjective(task_type='regression')
args = TrainingArguments(
    max_steps=500, learning_rate=5e-3, grad_clip_norm=1.0,
    early_stop_loss=0.01, keep_best=True, log_every=25,
)
history = Trainer(model, ds_train, objective, args).train()

# Evaluate on validation set
model.eval()
with torch.no_grad():
    output = model(x_val)
    loss = loss_mse(output, y_val, mask_val)
    acc = accuracy_general(output, y_val, mask_val)
print(f'Val loss: {loss:.4f}, Val acc: {acc:.4f}')

### 1.4 Inspecting Connectivity Vectors

For a rank-1 network, each neuron is characterized by 4 scalars:
$(m_i, n_i, w^{in}_i, w^{out}_i)$. The pairwise relationships reveal
how different components are organized across the population.

In [ ]:
# Extract vectors
m = model.m[:, 0].detach().numpy()
n = model.n[:, 0].detach().numpy()
wi = model.wi_full[0].detach().numpy()  # wi_full has per-channel scaling applied
wo = model.wo_full[:, 0].detach().numpy()

print(f'm: {m.shape}, n: {n.shape}, wi: {wi.shape}, wo: {wo.shape}')

# Pairwise scatter plots with linear regression
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
pairs = [(m, n, 'm', 'n'), (m, wi, 'm', 'w_in'), (m, wo, 'm', 'w_out'),
         (n, wi, 'n', 'w_in'), (n, wo, 'n', 'w_out'), (wi, wo, 'w_in', 'w_out')]
for i, (x, y, xl, yl) in enumerate(pairs):
    ax = axes[i // 3, i % 3]
    ax.scatter(x, y, alpha=0.4, s=10, c='steelblue')
    # Linear regression line
    if len(x) > 1:
        coeffs = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, np.polyval(coeffs, xs), 'r-', lw=2)
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(f'{xl} vs {yl}')
plt.tight_layout(); plt.show()

### 1.5 Trajectories in the m-Subspace

For rank-1, activity projected onto $m$ reveals a 1D decision variable.
The `return_dynamics=True` flag causes `forward()` to return a
`(output_tensor, trajectories_tensor)` tuple, with trajectories
containing $T+1$ time points (initial state prepended).

In [ ]:
coherences = [-4, -2, -1, 1, 2, 4]
colors = plt.cm.RdBu(np.linspace(0, 1, len(coherences)))
m_vec = model.m[:, 0].detach().numpy()
N = model.hidden_size

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for coh, color in zip(coherences, colors):
    inp, _, _, _, _, _ = generate_rdm_data(1, coherences=[coh], std=0.0)
    with torch.no_grad():
        out_tensor, traj = model(inp, return_dynamics=True)
        proj = traj[0].numpy() @ m_vec / N
    ax1.plot(proj, color=color, lw=2, label=f'coh={coh}')
    ax2.plot(out_tensor[0, :, 0], color=color, lw=2)
ax1.set_xlabel('Time step'); ax1.set_ylabel('Activity along m')
ax1.set_title('Hidden state projection')
ax2.set_xlabel('Time step'); ax2.set_ylabel('Output')
ax2.set_title('Readout')
ax1.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

### 1.6 Model Save/Load (NeuralRNN Style)

The framework uses `save_pretrained()` / `from_pretrained()` following
the HuggingFace convention: `config.json` + `model.safetensors`.

In [ ]:
os.makedirs('./models/07', exist_ok=True)
save_path = './models/07/tutorial_rdm_rank1'
model.save_pretrained(save_path)
print(f'Saved to {save_path}')
model2 = AutoModel.from_pretrained(save_path)
print(f'Loaded: {type(model2).__name__}')
with torch.no_grad():
    diff = (model(x_val[:5]).outputs - model2(x_val[:5]).outputs).abs().max().item()
print(f'Max output diff after load: {diff:.2e}')

---
## 2. Mante Task: Model Analysis

Context-dependent decision making: 4 inputs (color stim, motion stim, color ctx, motion ctx),
1 output. We train a **rank-1** RNN (4096 units) and reproduce the key figures from
Dubreuil et al. (2022).

### 2.1 Training

In [ ]:
hidden_size = 512
noise_std = 5e-2
alpha = 0.2

# Increase context scale for clearer separation (as in paper)
import neuralrnn.data.tasks.lr_mante_task as _mt
_mt.SCALE_CTX = 0.5
_mt._setup()  # re-derive timing globals

# Load Mante data using framework
ds_train = load_dataset('lr_mante', batch_size=64, num_trials=800, seed=42)
ds_val = load_dataset('lr_mante', batch_size=64, num_trials=200, seed=99)
x_val = ds_val.inputs
y_val = ds_val.targets
mask_val = ds_val.mask

cfg_mante = AutoConfig.for_model('lowrank_rnn',
    input_dim=4, latent_dim=hidden_size, output_dim=1,
    rank=1, alpha=alpha, noise_std=noise_std, activation='tanh',
)
model_mante = AutoModel.from_config(cfg_mante)
print(f'{model_mante}')
print(f'  params: {sum(p.numel() for p in model_mante.parameters()):,}')

# Train using framework Trainer (this will take some time with 4096 units)
objective = SupervisedObjective(task_type='regression')
args = TrainingArguments(
    max_steps=500, learning_rate=1e-2, grad_clip_norm=0.1,
    early_stop_loss=0.01, keep_best=True, log_every=50,
)
Trainer(model_mante, ds_train, objective, args).train()

In [ ]:
# Evaluate
model_mante.eval()
with torch.no_grad():
    output = model_mante(x_val)
    loss = loss_mse(output, y_val, mask_val)
    acc = accuracy_general(output, y_val, mask_val)
print(f'Val loss: {loss:.4f}, Val acc: {acc:.4f}')

### 2.2 Extract Vectors and Cluster Populations

Using the Bayesian GMM to find two functional populations in the
connectivity space.

In [ ]:
# Extract raw wi (without per-channel scaling) for clustering
m = model_mante.m[:, 0].detach().numpy()
n = model_mante.n[:, 0].detach().numpy()
wi1 = model_mante.wi[0].detach().numpy()
wi2 = model_mante.wi[1].detach().numpy()
wi_ctx1 = model_mante.wi[2].detach().numpy()
wi_ctx2 = model_mante.wi[3].detach().numpy()
wo = model_mante.wo[:, 0].detach().numpy()

# Make vectors for clustering
vecs = make_vecs(model_mante)

# Fit 2-population GMM
n_pops = 2
z, gmm_model = gmm_fit(vecs, n_pops, algo='bayes', n_init=50, random_state=2020)
z = 1 - z  # invert labels for presentation (as in original)

unique, counts = np.unique(z, return_counts=True)
for pop, cnt in zip(unique, counts):
    print(f'  Population {pop}: {cnt} neurons ({cnt/len(z)*100:.1f}%)')

# Adjust sign conventions for visualization
n = -n; m = -m

### 2.3 Population Scatter Plots (Figure 5b)

Each dot is a neuron; colors denote the GMM-assigned population.
Linear regressions show the relationship between connectivity vectors
within each population.

In [ ]:
colors_pop = ['seagreen', 'rebeccapurple']

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
pop_scatter_linreg(wi_ctx1, wi_ctx2, z, n_pops, colors=colors_pop,
                   linreg=False, ax=axes[0])
axes[0].set_title('Context inputs: color vs motion')
pop_scatter_linreg(n, wi1, z, n_pops, colors=colors_pop, ax=axes[1])
axes[1].set_title('n vs w_in (color)')
pop_scatter_linreg(n, wi2, z, n_pops, colors=colors_pop, ax=axes[2])
axes[2].set_title('n vs w_in (motion)')
plt.tight_layout(); plt.show()

### 2.4 Input/Output Visualization (Figure 4b-style)

Show example trials with colored stimulus periods.

In [ ]:
# Color scheme matching paper
col_ctx1 = '#840045'; col_ctx2 = '#ae8844'
col_sig1 = '#d3dd16'; col_sig2 = '#faa72a'
col_stim = 'silver'; col_resp = 'sandybrown'

def time_mapping(t):
    return t * _mante_deltaT / 1000

# Create two example trials with strong coherences
x_ex = torch.zeros((2, _mante_T, 4))
x_ex[:, :, :2] += _mante_STD * torch.randn((1, _mante_T, 2))
x_ex[:, _mante_stim_begin:_mante_stim_end, 0] += 2 * _mante_SCALE
x_ex[:, _mante_stim_begin:_mante_stim_end, 1] -= 1 * _mante_SCALE
x_ex[0, _mante_fixation:_mante_response_begin, 2] += _mante_SCALE_CTX
x_ex[1, _mante_fixation:_mante_response_begin, 3] += _mante_SCALE_CTX

with torch.no_grad():
    out_ex, traj_ex = model_mante(x_ex, return_dynamics=True)
    out_ex = out_ex.detach().numpy()
    traj_ex = traj_ex.detach().numpy().squeeze()

time = time_mapping(np.arange(x_ex.shape[1]))
x_np = x_ex.numpy().squeeze()

fig, axes = plt.subplots(3, 2, figsize=(12, 8))
# Input channel 1 (color stim)
axes[0,0].plot(time, x_np[0,:,0], lw=2, c=col_sig1)
axes[0,0].fill_between(
    time[_mante_stim_begin:_mante_stim_end], -0.3, 0.3,
    color=col_stim, alpha=0.3)
axes[0,0].set_title('Color stimulus (ctx=color trial)')
# Input channel 2 (motion stim)
axes[0,1].plot(time, x_np[0,:,1], lw=2, c=col_sig2)
axes[0,1].fill_between(
    time[_mante_stim_begin:_mante_stim_end], -0.3, 0.3,
    color=col_stim, alpha=0.3)
axes[0,1].set_title('Motion stimulus')
# Context inputs
axes[1,0].plot(time, x_np[0,:,2], lw=2, c=col_ctx1, label='ctx=color trial')
axes[1,0].plot(time, x_np[1,:,2], lw=2, c=col_ctx2, label='ctx=motion trial')
axes[1,0].set_title('Color context')
axes[1,0].legend(fontsize=8)
axes[1,1].plot(time, x_np[0,:,3], lw=2, c=col_ctx1)
axes[1,1].plot(time, x_np[1,:,3], lw=2, c=col_ctx2)
axes[1,1].set_title('Motion context')
# Output
axes[2,0].plot(time, out_ex[0,:,0], lw=2, c=col_ctx1, label='ctx=color')
axes[2,0].plot(time, out_ex[1,:,0], lw=2, c=col_ctx2, label='ctx=motion')
axes[2,0].fill_between(
    time[_mante_response_begin:], -1.5, 1.5,
    color=col_resp, alpha=0.2)
axes[2,0].set_title('Output'); axes[2,0].legend(fontsize=8)
axes[2,1].axis('off')
plt.tight_layout(); plt.show()

### 2.5 Psychometric Matrices (Figure 4c left)

Each cell shows the proportion of 'choice right' for a given (coh_color, coh_motion)
pair. Top panel: context = color. Bottom panel: context = motion.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(5, 8))
psychometric_matrices_mante(model_mante, ax=ax)
ax[0].set_title('Context = Color')
ax[1].set_title('Context = Motion')
plt.tight_layout(); plt.show()

### 2.6 Gain Analysis (Figure 5c)

The gain of each neuron ($\Phi'(x_i) = 1 - \tanh^2(x_i)$) determines
how much it contributes to the recurrent dynamics. Populations differ
in their average gain under different contexts.

In [ ]:
# Get trajectories for context=1 (color) and context=2 (motion)
x1, _, _, _, _, _ = generate_mante_data(
    1, coherences=[0.], std=0., context_spec=1)
with torch.no_grad():
    _, traj1 = model_mante(x1, return_dynamics=True)
    traj1 = traj1.detach().squeeze().numpy()
    phi_prime1 = phi_prime(
        traj1[_mante_stim_begin:_mante_stim_end].mean(axis=0))

x2, _, _, _, _, _ = generate_mante_data(
    1, coherences=[0.], std=0., context_spec=2)
with torch.no_grad():
    _, traj2 = model_mante(x2, return_dynamics=True)
    traj2 = traj2.detach().squeeze().numpy()
    phi_prime2 = phi_prime(
        traj2[_mante_stim_begin:_mante_stim_end].mean(axis=0))

fig, axes = plt.subplots(1, 2, figsize=(7, 4))
for ax, phi_p, title in [(axes[0], phi_prime1, 'Context = Color'),
                           (axes[1], phi_prime2, 'Context = Motion')]:
    vp = ax.violinplot([phi_p[z == i] for i in range(2)], showmeans=True)
    for i, body in enumerate(vp['bodies']):
        body.set_color(colors_pop[i]); body.set_alpha(1)
    ax.set_xticks([1, 2]); ax.set_xticklabels(['Pop. 1', 'Pop. 2'])
    ax.set_ylabel("Gain $\\Phi'(x_i)$"); ax.set_title(title)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

### 2.7 Effective Input per Population (Figure 5d)

The effective recurrent input $\kappa = n^T \cdot \tanh(W^{in} x)$
decomposed by population shows how each population contributes to the
overall dynamics.

In [ ]:
# Compute effective input for two trials
x = torch.zeros((2, _mante_T, 4))
x[:, _mante_stim_begin:_mante_stim_end, 0] += 2 * _mante_SCALE
x[:, _mante_stim_begin:_mante_stim_end, 1] -= 1 * _mante_SCALE
x[0, _mante_fixation:_mante_response_begin, 2] += _mante_SCALE_CTX
x[1, _mante_fixation:_mante_response_begin, 3] += _mante_SCALE_CTX

def compute_eff_input(x_trial, wi1, wi2, wi_ctx1, wi_ctx2, n, z, pop_idx):
    """Effective input for a specific population."""
    inp_integ = np.tanh(
        np.outer(x_trial[:, 0], wi1) + np.outer(x_trial[:, 1], wi2) +
        np.outer(x_trial[:, 2], wi_ctx1) + np.outer(x_trial[:, 3], wi_ctx2))
    pop_mask = (z == pop_idx)
    return inp_integ[:, pop_mask] @ n[pop_mask] / pop_mask.sum()

x_np = x.numpy().squeeze()
time = time_mapping(np.arange(x.shape[1]))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for trial_idx, ax in enumerate(axes):
    ctx_label = 'Color' if trial_idx == 0 else 'Motion'
    eff_pop1 = compute_eff_input(x_np[trial_idx], wi1, wi2, wi_ctx1, wi_ctx2, n, z, 0)
    eff_pop2 = compute_eff_input(x_np[trial_idx], wi1, wi2, wi_ctx1, wi_ctx2, n, z, 1)
    n_pop0 = (z==0).sum()
    n_pop1 = (z==1).sum()
    eff_all = (eff_pop1 * n_pop0 + eff_pop2 * n_pop1) / len(z)

    ax.plot(time, eff_pop1, c=colors_pop[0], lw=2, label='Pop 1')
    ax.plot(time, eff_pop2, c=colors_pop[1], lw=2, label='Pop 2')
    ax.plot(time, eff_all, c='firebrick', ls='--', lw=2, label='All')
    ax.set_title(f'Effective input (context={ctx_label})')
    ax.set_ylabel('$\\kappa$'); ax.set_xlabel('Time (s)')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

### 2.8 Overlap Matrices (Extended Data Fig 7c)

Cosine similarity between connection vectors within each population
reveals their functional organization.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, pop_idx in enumerate([0, 1]):
    pop_mask = (z == pop_idx)
    vectors = [wi1[pop_mask], wi2[pop_mask], wi_ctx1[pop_mask],
               wi_ctx2[pop_mask], n[pop_mask], m[pop_mask], wo[pop_mask] / 4]
    ov = overlap_matrix(vectors)
    labels = ['I^A', 'I^B', 'I^{cA}', 'I^{cB}', 'n', 'm', 'w']
    get_lower_tri_heatmap(ov, cbar=True, cbar_shrink=.5,
                          figsize=(4, 3), cbar_pad=.3, ax=axes[i])
    axes[i].set_xticks(np.arange(len(labels) - 1) + .5)
    axes[i].set_xticklabels(labels[1:])
    axes[i].set_yticks(np.arange(len(labels) - 1) + .5)
    axes[i].set_yticklabels(labels[:-1])
    axes[i].set_title(f'Population {pop_idx + 1}')
plt.tight_layout(); plt.show()

### 2.9 State-Space Trajectories (Extended Data Fig 8)

Project trajectories onto $(m, I^A_\perp)$ and $(m, I^B_\perp)$ planes
for all coherence combinations. This section has been **rewritten** to
work correctly with arbitrary hidden sizes (no hard-coded population dimensions).

In [ ]:
# Compute orthogonal input directions (with sign flip for presentation)
Ia_orth = wi1 - (wi1 @ m) / (m @ m) * m
Ib_orth = wi2 - (wi2 @ m) / (m @ m) * m

# Generate 32 trials (16 ctx=1, 16 ctx=2) directly — no population-group dimension
n_trials = 32
c1, c2 = 0.5, 1.0
x_si = torch.zeros((n_trials, _mante_T, 4))

# ctx=1 trials (0-15): attend color
for grp_offset, coh_a_sign in [(0, 1), (4, 1), (8, -1), (12, -1)]:
    coh_a_val = c1 * _mante_SCALE if grp_offset in [0, 8] else c2 * _mante_SCALE
    for j in range(4):
        t = grp_offset + j
        x_si[t, _mante_stim_begin:_mante_stim_end, 0] = coh_a_sign * coh_a_val
        coh_b_vals = [c1*_mante_SCALE, c2*_mante_SCALE,
                      -c1*_mante_SCALE, -c2*_mante_SCALE]
        x_si[t, _mante_stim_begin:_mante_stim_end, 1] = coh_b_vals[j]
        x_si[t, _mante_fixation:_mante_response_begin, 2] = _mante_SCALE_CTX

# ctx=2 trials (16-31): attend motion
for grp_offset, coh_b_sign in [(16, 1), (20, 1), (24, -1), (28, -1)]:
    coh_b_val = c1 * _mante_SCALE if grp_offset in [16, 24] else c2 * _mante_SCALE
    for j in range(4):
        t = grp_offset + j
        x_si[t, _mante_stim_begin:_mante_stim_end, 1] = coh_b_sign * coh_b_val
        coh_a_vals = [c1*_mante_SCALE, c2*_mante_SCALE,
                      -c1*_mante_SCALE, -c2*_mante_SCALE]
        x_si[t, _mante_stim_begin:_mante_stim_end, 0] = coh_a_vals[j]
        x_si[t, _mante_fixation:_mante_response_begin, 3] = _mante_SCALE_CTX

# Forward pass: get rates (tanh of hidden states)
with torch.no_grad():
    _, traj_si = model_mante(x_si, return_dynamics=True)
    rates_si = np.tanh(traj_si.detach().numpy())  # (n_trials, T+1, N)

# Project onto m and orthogonal input directions
N_h = model_mante.hidden_size
proj_m = rates_si @ m / N_h       # (n_trials, T+1)
proj_Ia = rates_si @ Ia_orth / N_h  # (n_trials, T+1)
proj_Ib = rates_si @ Ib_orth / N_h  # (n_trials, T+1)

print(f'rates_si: {rates_si.shape} → proj_m: {proj_m.shape}, proj_Ia: {proj_Ia.shape}')

In [ ]:
# Plot (m, Ia_orth) for ctx=1 trials
fig, ax = plt.subplots(figsize=(6, 6))
colors_si = ['firebrick', 'royalblue', 'green', 'blueviolet']

# Group mean trajectories for different coherence sign/strength combos
groups_ctx1 = [(0, 4), (4, 8), (8, 12), (12, 16)]
alphas = [0.5, 1.0, 0.5, 1.0]
for idx, ((start, end), alpha) in enumerate(zip(groups_ctx1, alphas)):
    color_idx = 0 if idx < 2 else 1  # pos coh A: firebrick, neg coh A: royalblue
    mean_m = proj_m[start:end].mean(axis=0)
    mean_Ia = proj_Ia[start:end].mean(axis=0)
    ax.plot(mean_m, mean_Ia, c=colors_si[color_idx], alpha=alpha, lw=2)

# Direction vectors
m_scale = (m @ m) / N_h
Ia_scale = (Ia_orth @ Ia_orth) / N_h
wi1_m = (wi1 @ m) / (m @ m)
wi1_Ia = (wi1 @ Ia_orth) / (Ia_orth @ Ia_orth)

ax.quiver(0, 0, 1, 0, scale=8, color='goldenrod', width=0.005)
ax.quiver(0, 0, wi1_m, wi1_Ia, scale=8, color='green', width=0.005)

# Compute text positions based on data range
mx, my = proj_m[:, :16].mean(), proj_Ia[:, :16].mean()
dx = proj_m.ptp() / 4
ax.text(mx + dx, my, '$\\mathbf{I}^A$', size=16, color='green')
ax.text(mx + dx * 2, my - dx, '$\\mathbf{m}$', size=16, color='goldenrod')

ax.set_title('State-space: (m, I_A_orth) — Context=Color')
ax.set_axis_off()
plt.tight_layout(); plt.show()

### 2.10 Save Mante Model

In [ ]:
save_path = './models/07/mante_rank1'
model_mante.save_pretrained(save_path)
print(f'Mante model saved to {save_path}')

---
## 3. Quick Training Check on All 5 Tasks

Training a rank-1 RNN on each of the 5 Dubreuil et al. (2022) tasks
to verify that our neuralrnn model architecture achieves near-ceiling accuracy.

In [ ]:
def quick_train(name, task_name, input_dim, hidden, output_dim, rank,
                epochs, lr, clip_val, batch_size=32, num_trials=800):
    """Train a rank-1 low-rank RNN on a task using the framework Trainer."""
    print(f'\n{"="*50}')
    print(f'{name}: input={input_dim}, hidden={hidden}, rank={rank}')
    
    # Load data via framework
    ds = load_dataset(task_name, batch_size=batch_size, num_trials=num_trials, seed=42)
    # Use a separate validation set
    ds_val = load_dataset(task_name, batch_size=batch_size, num_trials=max(100, num_trials//5), seed=99)
    x_v, y_v, m_v = ds_val.inputs, ds_val.targets, ds_val.mask
    
    cfg = AutoConfig.for_model('lowrank_rnn',
        input_dim=input_dim, latent_dim=hidden, output_dim=output_dim,
        rank=rank, alpha=0.2, noise_std=0.05, activation='tanh')
    model = AutoModel.from_config(cfg)
    
    max_steps = epochs * (num_trials // batch_size)
    objective = SupervisedObjective(task_type='regression')
    args = TrainingArguments(max_steps=max_steps, learning_rate=lr,
                             grad_clip_norm=clip_val, keep_best=True, log_every=0)
    Trainer(model, ds, objective, args).train()
    
    model.eval()
    with torch.no_grad():
        output = model(x_v)
        loss = loss_mse(output, y_v, m_v)
        acc = accuracy_general(output, y_v, m_v)
    print(f'  {name}: loss={loss:.4f}, acc={acc:.4f}')
    return model, loss.item(), acc.item()

results = {}

In [ ]:
# 1. RDM
model_rdm, _, acc_rdm = quick_train(
    'RDM', 'rdm', 1, 256, 1, 1,
    20, 5e-3, 1.0, num_trials=800)
results['RDM'] = acc_rdm

# 2. Romo
model_romo, _, acc_romo = quick_train(
    'Romo', 'romo', 1, 256, 1, 1,
    40, 5e-3, 1.0, num_trials=800)
results['Romo'] = acc_romo

# 3. Raposo
model_raposo, _, acc_raposo = quick_train(
    'Raposo', 'raposo', 4, 256, 1, 1,
    50, 5e-3, 1.0, num_trials=800)
results['Raposo'] = acc_raposo

# 4. Mante
model_mante_q, _, acc_mante_q = quick_train(
    'Mante', 'lr_mante', 4, 512, 1, 1,
    50, 3e-3, 0.1, num_trials=1000)
results['Mante'] = acc_mante_q

# 5. DMS
model_dms_q, _, acc_dms_q = quick_train(
    'DMS', 'dms', 2, 256, 1, 1,
    60, 5e-3, 1.0, num_trials=800)
results['DMS'] = acc_dms_q

In [ ]:
# Summary
print('\n' + '='*50)
print('SUMMARY')
print('='*50)
for name, acc_val in results.items():
    status = 'PASSED' if acc_val > 0.85 else (
        'MARGINAL' if acc_val > 0.7 else 'FAILED')
    print(f'  {name:10s}: accuracy = {acc_val:.4f}  [{status}]')
print('='*50)

---
## 4. Rank Comparison: Oscillator Dataset

Using the oscillator dataset from `03_custom_pipeline.ipynb`, compare
rank 1, 2, and 3 low-rank RNNs on a rhythmic output task.

### 4.1 Generate Oscillator Dataset

Input: constant level starting at random onset (1–2 s).  
Output: sine wave at the input's frequency (`level` Hz), starting at onset.

In [ ]:
def generate_oscillator(n_trials=200, dt=0.01, duration=5.0,
                        onset_min=1.0, onset_max=2.0, levels=(0.2, 0.4),
                        noise_std=0.001, seed=42):
    rng = np.random.RandomState(seed)
    n_steps = int(duration / dt)
    inputs = np.zeros((n_trials, n_steps, 1))
    targets = np.zeros((n_trials, n_steps, 1))
    mask = np.zeros((n_trials, n_steps, 1))
    conditions = []
    for i in range(n_trials):
        level = levels[rng.randint(0, len(levels))]
        onset_step = int(rng.uniform(onset_min, onset_max) / dt)
        inputs[i, onset_step:, 0] = level
        t_post = np.arange(n_steps - onset_step) * dt
        targets[i, onset_step:, 0] = np.sin(2 * np.pi * level * t_post)
        mask[i, onset_step:, 0] = 1.0
        inputs[i] += noise_std * rng.randn(*inputs[i].shape)
        conditions.append({'level': level, 'onset': onset_step})
    return (torch.tensor(inputs, dtype=torch.float32),
            torch.tensor(targets, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32), conditions)

osc_in, osc_tgt, osc_msk, osc_cond = generate_oscillator(n_trials=200, seed=42)
split = int(0.8 * len(osc_in))
osc_tr = (osc_in[:split], osc_tgt[:split], osc_msk[:split])
osc_vl = (osc_in[split:], osc_tgt[split:], osc_msk[split:])
print(f'Oscillator: {osc_in.shape}, train={split}, val={len(osc_in)-split}')

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 5))
for idx, (ax_in, ax_out) in enumerate([axes[0], axes[1]]):
    trial = idx * 50
    onset = osc_cond[trial]['onset']
    ax_in.plot(osc_in[trial, :, 0], 'b', lw=1)
    ax_in.axvline(onset, color='orange', ls='--')
    ax_in.set_ylabel(f'Input (level={osc_cond[trial]["level"]})')
    ax_out.plot(osc_tgt[trial, :, 0], 'r', lw=1)
    ax_out.axvline(onset, color='orange', ls='--')
    ax_out.set_ylabel(f'Target ({osc_cond[trial]["level"]} Hz)')
plt.tight_layout(); plt.show()

### 4.2 Train Rank 1, 2, 3

In [ ]:
def train_osc(rank, hidden=256, epochs=1000):
    """Train a low-rank RNN on the oscillator dataset using framework Trainer."""
    x_tr, y_tr, m_tr = osc_tr
    # Build dataset from tensors
    ds = CognitiveTaskDataset(x_tr, y_tr, m_tr, conditions=[], batch_size=32)
    
    cfg = AutoConfig.for_model('lowrank_rnn',
        input_dim=1, latent_dim=hidden, output_dim=1,
        rank=rank, alpha=0.2, noise_std=0, activation='tanh')
    model = AutoModel.from_config(cfg)
    
    max_steps = epochs * (len(x_tr) // 32)
    objective = SupervisedObjective(task_type='regression')
    args = TrainingArguments(max_steps=max_steps, learning_rate=5e-3,
                             grad_clip_norm=1.0, keep_best=True, log_every=20)
    history = Trainer(model, ds, objective, args).train()
    
    model.eval()
    x_v, y_v, m_v = osc_vl
    with torch.no_grad():
        out = model(x_v)
        err = (out.outputs - y_v) ** 2
        vloss = (err * m_v).sum() / m_v.sum().clamp_min(1.0)
    
    # Extract loss history
    loss_history = [h['loss'] for h in history]
    if len(loss_history) < epochs:
        # Pad if early stopped
        loss_history = loss_history + [loss_history[-1]] * (epochs - len(loss_history))
    return model, loss_history, vloss.item()

print('Training rank 1, 2, 3...')
mr1, hr1, vl1 = train_osc(1)
mr2, hr2, vl2 = train_osc(2)
mr3, hr3, vl3 = train_osc(3)

### 4.3 Training Curves and Predictions

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(hr1, label=f'Rank 1 (vl={vl1:.4f})', lw=2)
ax1.plot(hr2, label=f'Rank 2 (vl={vl2:.4f})', lw=2)
ax1.plot(hr3, label=f'Rank 3 (vl={vl3:.4f})', lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training curves')
ax1.legend(); ax1.set_yscale('log')

# Predictions for one trial
trial = 0; x_v, y_v, _ = osc_vl
with torch.no_grad():
    ax2.plot(y_v[trial, :, 0], 'k', lw=2, alpha=0.5, label='Target')
    for model, label, color in [(mr1, 'R1', 'b'), (mr2, 'R2', 'g'),
                                 (mr3, 'R3', 'r')]:
        ax2.plot(model(x_v[trial:trial+1]).outputs[0, :, 0],
                 color=color, lw=1, label=label)
ax2.set_xlabel('Time step'); ax2.set_ylabel('Output')
ax2.set_title('Predictions vs Target'); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f'Rank 1: {sum(p.numel() for p in mr1.parameters()):,} params, val_loss={vl1:.4f}')
print(f'Rank 2: {sum(p.numel() for p in mr2.parameters()):,} params, val_loss={vl2:.4f}')
print(f'Rank 3: {sum(p.numel() for p in mr3.parameters()):,} params, val_loss={vl3:.4f}')

### 4.4 PCA Trajectory Visualization

In [ ]:
def plot_pca(model, label, ax):
    x_v, _, _ = osc_vl
    states_all = []
    colors_all = []
    for level, col in [(0.2, 'blue'), (0.4, 'red')]:
        for i in range(len(x_v)):
            if osc_cond[split + i]['level'] == level:
                with torch.no_grad():
                    out = model(x_v[i:i+1], return_dynamics=True)
                    _, traj = out
                    states_all.append(traj[0].numpy())
                    colors_all.append(col)
                break
    all_s = np.concatenate(states_all, axis=0)
    pca = PCA(n_components=2).fit(all_s)
    for states, col in zip(states_all, colors_all):
        proj = pca.transform(states)
        ax.plot(proj[:, 0], proj[:, 1], color=col, lw=1.5, alpha=0.8)
    ax.set_title(label)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_pca(mr1, 'Rank 1', axes[0])
plot_pca(mr2, 'Rank 2', axes[1])
plot_pca(mr3, 'Rank 3', axes[2])
for ax in axes: ax.legend(['0.2 Hz', '0.4 Hz'], fontsize=8)
plt.tight_layout(); plt.show()

---
## 5. Summary

This notebook demonstrated the full low-rank RNN pipeline **using only the neuralrnn framework**
(all task generators, model architecture, and save/load).

| Section | Content | Key APIs |
|---------|---------|----------|
| 1 | Tutorial: Rank-1 RDM | `LowrankRNNConfig`, `train()`, `save_pretrained()` |
| 2 | Mante full analysis | `psychometric_matrices_mante()`, `gmm_fit()`, `pop_scatter_linreg()`, gain, effective input, overlap, state-space |
| 3 | All 5 tasks | RDM, Romo, Raposo, DMS, Mante — accuracy check |
| 4 | Rank comparison | Rank 1/2/3 on oscillator dataset, PCA |

**Key architectural decisions:**
- Framework-standard naming: `activation` (not `non_linearity`), `latent_dim` (aliased as `hidden_size`)
- `forward(return_dynamics=True)` returns `(output, trajectories)` tuple
- `clone()` method for `train()` best-model tracking
- Contiguous tensor handling in `save_pretrained()`
- All task data generated through `neuralrnn.data.tasks`
- Analysis/plotting helpers defined locally in notebook

**References:**
- Dubreuil et al. (2022) *Nature Neuroscience*
- Valente et al. (2022) *NeurIPS*